# TensorFlow: Using Grad-CAM approach to analyze CNN

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

In [ ]:
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "1"

import tensorflow as tf
import tensorflow_datasets as tfds
import tensorflow.keras.backend as K
from tensorflow.keras.applications.vgg16 import VGG16

print("TF Version: ", tf.__version__)
print("TF Eager mode: ", tf.executing_eagerly())
print("TF GPU is", "available" if tf.config.list_physical_devices("GPU") else "not available")

In [ ]:
(raw_tr_ds, raw_vl_ds, raw_ts_ds), ds_info = tfds.load(
    'cats_vs_dogs',
    with_info=True,
    as_supervised=True,
    split=['train[:80%]', 'train[80%:90%]', 'train[90%:]'])

In [ ]:
n_examples = ds_info.splits['train'].num_examples
n_classes = ds_info.features['label'].num_classes

In [ ]:
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

def format_image(image, label):
    image = tf.image.resize(image, IMAGE_SIZE) / 255.0
    return  image, label

tr_ds = (raw_tr_ds
         .shuffle(n_examples // 4)
         .map(format_image)
         .batch(BATCH_SIZE)
         .prefetch(tf.data.experimental.AUTOTUNE))
vl_ds = (raw_vl_ds
         .map(format_image)
         .batch(BATCH_SIZE)
         .prefetch(tf.data.experimental.AUTOTUNE))
ts_ds = (raw_ts_ds
         .map(format_image)
         .batch(1))

In [ ]:
def build_model():
  # Load the base VGG16 model
  base_model = VGG16(input_shape=IMAGE_SIZE + (3,),
                     weights='imagenet',
                     include_top=False)
  # Add a GAP layer
  output = tf.keras.layers.GlobalAveragePooling2D()(base_model.output)
  # Output has two neurons for the 2 classes (cats and dogs)
  output = tf.keras.layers.Dense(2, activation='softmax')(output)
  # Set the inputs and outputs of the model
  model = tf.keras.Model(base_model.input, output)
  # Freeze the earlier layers
  for layer in base_model.layers[:-4]:
      layer.trainable=False
  # Choose the optimizer
  optimizer = tf.keras.optimizers.RMSprop(0.001)
  # Configure the model for training
  model.compile(loss='sparse_categorical_crossentropy',
                optimizer=optimizer,
                metrics=['accuracy'])
  return model

In [ ]:
model = build_model()

In [ ]:
EPOCHS = 3
_ = model.fit(tr_ds,
          epochs=EPOCHS,
          validation_data=vl_ds)

## Grad-CAM

In [ ]:
def get_CAM(model, image, label, layer_name='block5_conv3', negate=False, debug=False):
    with tf.GradientTape() as tape:
        activations, predictions = model(image)
        # Watch the conv_output_values
        tape.watch(activations)
        # Calculate loss as binary cross entropy
        #   y_true.shape => [1, ]
        #   y_pred.shape => [1, n_classes]
        loss = tf.keras.losses.sparse_categorical_crossentropy(
            np.array([tf.cast(label, dtype=tf.float32)]),
            predictions)

    print(f'Loss: {loss}')

    # Get the gradients of the scope w.r.t. feature map activation
    grads_values = tape.gradient(loss, activations)
    # Obtain the neuron importance weights by applying global-average-ppoling
    if negate is True:
        grads_values = tf.reduce_mean(-1 * grads_values, axis=(0,1,2))
    else:
        grads_values = tf.reduce_mean(grads_values, axis=(0,1,2))

    activations = np.squeeze(activations.numpy())
    if debug is True:
        print(f'Activations of "{layer_name}" layer shape: {activations.shape}')
    grads_values = grads_values.numpy()
    if debug is True:
        print(f'Grads of "{layer_name}" layer shape: {grads_values.shape}')

    # Weight the convolution outputs with the computed gradients
    activations = np.multiply(activations, grads_values)

    # Final post-processing stage:
    # - channel-wise averaging (flattens the depth by taking the average across channel dimension)
    # - applying manual ReLU (we interested in only the positive average activations)
    # - identify the most important 1x1 area for a model (decision maker)
    # - apply min-max normalization to scale heatmap to a range [0,1]
    heatmap = np.mean(activations, axis=-1)
    heatmap = np.maximum(heatmap, 0)
    global_max = np.max(heatmap)
    heatmap = np.divide(heatmap,
                        global_max,
                        out=np.zeros_like(heatmap),
                        where=global_max!=0)
    return heatmap

In [ ]:
def overlay_heatmap(image, heatmap):
    # Preprocess heatmap
    heatmap = cv2.resize(heatmap, (test_image.shape[0], test_image.shape[1]))
    heatmap = heatmap * 255
    heatmap = np.clip(heatmap, 0, 255).astype(np.uint8)
    heatmap = cv2.applyColorMap(heatmap, cv2.COLORMAP_HOT)
    # Clip the image values
    image = np.clip(cv2.addWeighted(image, 0.8, heatmap.astype('float32'), 2e-3, 0.0), 0.0, 1.0)
    return image

In [ ]:
layer_name = 'block5_conv3'

grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[model.get_layer(layer_name).output, model.output])

In [ ]:
(test_image, test_label) = next(iter(ts_ds.shuffle(1000).take(1)))
test_image = test_image[0]
test_label = test_label[0]
print(f'Test image shape <{test_image.shape}> with label <{test_label}>')

# Add batch dimension
batched_image = np.expand_dims(test_image, axis=0)

# Standard Grad-CAM
# "I think this is a Husky because of these pointed ears."
heatmap_p = get_CAM(grad_model,
    image=batched_image,
    label=test_label,
    negate=False,
    debug=True)
# Counterfactual Grad-CAM
# "I would be more sure this is a Husky if that fluffy cat tail in the corner wasn't there."
heatmap_n = get_CAM(grad_model,
    image=batched_image,
    label=test_label,
    negate=True,
    debug=True)

overlay_image_p = overlay_heatmap(test_image.numpy(), heatmap_p)
overlay_image_n = overlay_heatmap(test_image.numpy(), heatmap_n)

fig, ax = plt.subplots(2, 2)
ax[0][0].imshow(heatmap_p)
ax[0][0].axis('off')
ax[0][1].imshow(overlay_image_p)
ax[0][1].axis('off')
ax[1][0].imshow(heatmap_n)
ax[1][0].axis('off')
ax[1][1].imshow(overlay_image_n)
ax[1][1].axis('off')
plt.tight_layout()
plt.show()

## Multi-CAM

In [ ]:
def visualize_intermediate_activations(layer_names, activations):
    assert len(layer_names)==len(activations), "Make sure layers and activation values match"
    images_per_row=16

    for layer_name, layer_activation in zip(layer_names, activations):
        nb_features = layer_activation.shape[-1]
        size= layer_activation.shape[1]

        nb_cols = nb_features // images_per_row
        grid = np.zeros((size*nb_cols, size*images_per_row))

        for col in range(nb_cols):
            for row in range(images_per_row):
                feature_map = layer_activation[0,:,:,col*images_per_row + row]
                feature_map -= feature_map.mean()
                feature_map /= feature_map.std()
                feature_map *=255
                feature_map = np.clip(feature_map, 0, 255).astype(np.uint8)

                grid[col*size:(col+1)*size, row*size:(row+1)*size] = feature_map

        scale = 1./size
        plt.figure(figsize=(scale*grid.shape[1], scale*grid.shape[0]))
        plt.title(layer_name)
        plt.grid(False)
        plt.axis('off')
        plt.imshow(grid, aspect='auto', cmap='viridis')
    plt.show()

In [ ]:
# Select all the layers to visualize the outputs
outputs = [layer.output for layer in model.layers[1:18]]

# Define a new model that generates the above output
vis_model = tf.keras.Model(model.input, outputs)

# store the layer names we are interested in
layer_names = []
for layer in outputs:
    layer_names.append(layer.name.split("/")[0])
print('Layers that will be used for visualization: ')
print(layer_names)

In [ ]:
index = None
sample_image = None
sample_label = None

if index:
    for image, label in ts_ds.take(index):
        sample_image = np.expand_dims(image[0], axis=0)
        sample_label = label[0]
else:
    for image, label in ts_ds.shuffle(1000).take(1):
        sample_image = np.expand_dims(image[0], axis=0)
        sample_label = label[0]

activations = vis_model.predict(sample_image)

print(f'Activations shape: {len(activations)}')
for name, acts in zip(layer_names, activations):
    if acts is not None:
        print(f'...Layer<{name}> has activations shape<{acts.shape}>')

In [ ]:
visualize_intermediate_activations(activations=activations,
                                   layer_names=layer_names)